## RAG(Retrieval Augmented Generation)

0. **기존 LLM의 한계점**
- 환각현상: LLM은 트렌스포머의 Decoder 모듈이 여러개 결합된 모델로, 텍스트를 생성하는 기능에 특화되어 있다. 즉, 어떤 문장이 주어지더라도 어떻게 해서든 답변을 하도록 훈련이 되어있다.
- 컨텍스트 윈도우 제한 문제: 입력으로 주어진 텍스트 값의 길이에 상한선이 존재
 

1. **검색-증강-생성(RAG)란**<br> 
RAG(Retrieval Augmented Generation)는 생성형 AI와 정보 검색 기술을 결합한 방식으로, 복잡한 질문이나 최신 정보에 대해 더욱 정확하고 신뢰성 있는 답변을 제공한다. RAG시스템은 사용자 질문에 답변하기 위해 LLM에만 의존하는 것이 아니라 외부 지식을 참고하도록 만들어진 기술이라고 할 수 있다.<br>

2. **RAG의 작동 단계** <br>
- 크게 "**정보 저장(인덱싱)**", "**검색**, **생성**"의 단계로 나눌 수 있다.

1) 외부 데이터 생성: API, 데이터베이스, 문서 레포지토리 등에서 외부 데이터를 수집한 후, 임베딩 언어 모델 을 통해 벡터 데이터로 변환하여 벡터 데이터베이스에 저장
2) 정보 검색: 사용자가 입력한 쿼리를 벡터로 변환하고, 이를 벡터 데이터베이스와 비교하여 관련성이 높은 문서나 정보를 검색
3) LLM 프롬프트 확장: 검색된 정보는 LLM에 전달되어 기존 학습 데이터에 추가적인 맥락을 제공
4) 응답 생성: LLM은 검색된 정보를 기반으로 확장된 프롬프트를 이용해 더욱 정확하고 관련성 있는 응답을 생성
5) 외부 데이터 업데이트: 외부 데이터는 실시간으로 또는 주기적으로 업데이트되며, 벡터 데이터베이스 역시 최신 정보로 갱신

(참고:https://aws.amazon.com/ko/what-is/retrieval-augmented-generation/)

![RAG](images/image-12.png)


> 이번 챕터에서 다룰 파트 : **정보 저장(인덱싱)** <br>

RAG는 사전에 정보를 가공하여 **벡터 데이터베이스**(Vector 저장소)에 저장해 두고, 나중에 검색할 수 있도록 준비한다. 이 단계는 다음과 같은 과정으로 이루어진다.

1. **Load (불러오기)**
   - 답변시 참조할 사전 정보를 가진 데이터들을 불러온다.
2. **Split/Chunking (문서 분할)**
   - 긴 텍스트를 일정한 길이의 작은 덩어리(*chunk*)로 나눈다.
   - 이렇게 해야 검색과 생성의 정확도를 높일 수 있다.
3. **Embedding (임베딩)**
   - 각 텍스트 조각을 **임베딩 벡터**로 변환한다.
   - 임베딩 벡터는 그 문서의 의미를 벡터화 한 것으로 질문과 유사한 문서를 찾을 때 인덱스로 사용된다.
4. **Store (저장)**
   - 임베딩된 벡터를 **벡터 데이터베이스**(벡터 저장소)에 저장한다.
   - 벡터 데이터베이스는 유사한 질문이나 문장을 빠르게 찾을 수 있도록 특화된 데이터 저장소이다.

![rag](images/rag1.png)

In [1]:
# 0. API 연결(load_dotenv) -> 이 정도는 외울 수 있자나~!
from dotenv import load_dotenv

load_dotenv()

True

### 1. load&split (Chunking) : RecursiveCharacterTextSplitter
- RecursiveCharacterTextSplitter는 **긴 텍스트를 지정된 최대 길이(chunk_size) 이하로 나누는 데 효과적인 텍스트 분할기**(splitter)이다.
- 여러 **구분자(separators)를 순차적으로 적용**하여, 가능한 한 자연스러운 문단/문장/단어 단위로 분할하고, 최종적으로는 크기 제한을 만족시킨다.
- 분할 기준 문자
    1. 두 개의 줄바꿈 문자 ("\n\n")
    2. 한 개의 줄바꿈 문자 ("\n")
    3. 공백 문자 (" ")
    4. 빈 문자열 ("")

- 주요 파라미터
    - chunk_size: 각 조각의 최대 길이를 지정.
    - chunk_overlap: 연속된 청크들 간의 겹치는 문자 수를 설정. 새로운 청크 생성 시 이전 청크의 마지막 부분에서 지정된 수만큼의 문자를 가져와서 새 청크의 앞부분에 포함시켜, 청크 경계에서 문맥의 연속성을 유지한다.
      - 구분자에 의해 청크가 나눠지면 정상적인 분리이므로 overlap이 적용되지 않는다.
      - 정상적 구분자로 나눌 수 없어 chunk_size에 맞춰 잘라진 경우 문맥의 연결성을 위해 overlap을 적용한다.
    - separators(list): 구분자를 지정한다. 지정하면 기본 구분자가 지정한 것으로 변경된다.

In [ ]:
# 1. text loading
from langchain_community.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

path = "data/olympic.txt"
loader = TextLoader(path, encoding = "utf-8")
docs = loader.load()   

# 2.load한 문서를 Split
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap= 100)
split_docs = splitter.split_documents(docs)

print(len(split_docs))

61


'올림픽'

In [3]:
# 3. embedding 모델 생성 
from langchain_openai import OpenAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

In [8]:
# 4. Document를 활용해서 id를 직접 만들어보기
from langchain_core.documents import Document
docs =[Document(id=f"{i+1}", page_content= split_docs[i].page_content, metadata={'source': 'data/olympic.txt'}) for i in range(len(split_docs))]
docs[0].page_content

'올림픽'

### Ver.1 : Embedding + cosine similarity(embedded_query와 embedded_docs 간의 유사도 계산)
- **`embed_documents(texts: List[str])`**
    - 여러 문서를 받아 벡터화(임베딩)한다.
    - Context를 벡터화 할 때 사용한다.
- **`embed_query(text: str)`**
    - 하나의 문자열(문서)을 받아 벡터화한다.
    - Query를 벡터화 할 때 사용한다.

In [12]:
docs_content = [docs[i].page_content for i in range(61)]
embedded_docs = embedding_model.embed_documents(docs_content)

In [13]:
import numpy as np
np.shape(embedded_docs)

(61, 1536)

In [14]:
# 임베딩 벡터로 변환 -> 유사도 계산 -> 유사도 높은 순으로 출력

query = "근대 올림픽의 역사와 국제 올림픽 위원회(IOC)의 역할에 대해 설명하라."
embedded_query = embedding_model.embed_query(query)   # 임베딩 벡터로 변환
np.shape(embedded_query), type(embedded_query)

((1536,), list)

In [15]:
# 코사인 유사도(https://benn.tistory.com/62)
import numpy as np


def cosine_similarity(v1:np.ndarray, v2:np.ndarray|list)-> float:
    # v1 과 v2의 코사인 유사도 계산
    # -1~1
    # 1: 같은 것 0:관계없는 것 # -1: 반대
    v1= np.array(v1)
    v2= np.array(v2)
    return (v1 @ v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))

In [22]:
# embedded_query와 embedded_docs 간의 유사도 계산
for i, ev in enumerate(embedded_docs):
    print(f"{i+1}. {cosine_similarity(ev, embedded_query)}, {docs_content[i]}")

1. 0.38748610257637767, 올림픽
2. 0.5381842175257993, 올림픽(영어: Olympic Games, 프랑스어: Jeux olympiques)은 전 세계 각 대륙 각국에서 모인 수천 명의 선수가 참가해 여름과 겨울에 스포츠 경기를 하는 국제적인 대회이다. 전 세계에서 가장 큰 지구촌 최대의 스포츠 축제인 올림픽은 세계에서 가장 인지도있는 국제 행사이다. 올림픽은 2년마다 하계 올림픽과 동계 올림픽이 번갈아 열리며, 국제 올림픽 위원회(IOC)가 감독하고 있다. 또한 오늘날의 올림픽은 기원전 8세기부터 서기 5세기에 이르기까지 고대 그리스 올림피아에서 열렸던 올림피아 제전에서 비롯되었다. 그리고 19세기 말에 피에르 드 쿠베르탱 남작이 고대 올림피아 제전에서 영감을 얻어, 근대 올림픽을 부활시켰다. 이를 위해 쿠베르탱 남작은 1894년에 IOC를 창설했으며, 2년 뒤인 1896년에 그리스 아테네에서 제 1회 올림픽이 열렸다. 이때부터 IOC는 올림픽 운동의 감독 기구가 되었으며, 조직과 활동은 올림픽 헌장을 따른다. 오늘날 전 세계 대부분의
3. 0.4073350638593863, 1896년에 그리스 아테네에서 제 1회 올림픽이 열렸다. 이때부터 IOC는 올림픽 운동의 감독 기구가 되었으며, 조직과 활동은 올림픽 헌장을 따른다. 오늘날 전 세계 대부분의 국가에서 올림픽 메달은 매우 큰 영예이며, 특히 올림픽 금메달리스트는 국가 영웅급의 대우를 받으며 스포츠 스타가 된다. 국가별로 올림픽 메달리스트들에게 지급하는 포상금도 크다. 대부분의 인기있는 종목들이나 일상에서 쉽게 접하고 즐길 수 있는 생활스포츠 종목들이 올림픽이라는 한 대회에서 동시에 열리고, 전 세계 대부분의 국가 출신의 선수들이 참여하는 만큼 전 세계 스포츠 팬들이 가장 많이 시청하는 이벤트이다. 2008 베이징 올림픽의 모든 종목 누적 시청자 수만 47억 명에 달하며, 이는 인류 역사상 가장 많은 수의 인구가 시청한 이벤트였다.
4. 0.5374869012578062, 

### Ver.2 : 벡터 데이터베이스(Vector Database)
- **벡터 데이터베이스의 주요 기능**
1. **저장**  
   - 이미지, 텍스트, 음성 등 **비정형 데이터**를 임베딩 모델을 통해 벡터로 변환한 뒤 벡터 데이터베이스에 저장한다.
2. **검색**  
   - 검색하려는 데이터를 임베딩 모델로 변환한 뒤, 벡터 데이터베이스에서 **유사도를 기반**으로 검색한다.
   > 임베딩 모델에 따라 성능의 차이를 보인다. 데이터가 많아질수록 유사도 계산이 오래 걸리기 때문에, 모델들마다 계산 관련 알고리즘이 있다. 그 알고리즘의 성능에 따라 유사도 성능에 차이를 보일 수 있음.
3. **결과 반환**  
   - 벡터 데이터베이스는 저장된 벡터 중 검색 쿼리 임베딩과 가장 가까운 벡터를 찾아 반환한다.


In [19]:
# DB를 연결하면서 document를 upsert -> vector store(InMemoryVectorStore)에 저장
vector_store = InMemoryVectorStore.from_documents(
    documents = docs,
    embedding = embedding_model
)

In [20]:
# 유사도 계산 (1.cosine_similarity, 2. vector_store.silmarity_search_with_score()) 
# Query와 유사한 문서를 Vector Store에서 찾기.
query = "근대 올림픽의 역사와 국제 올림픽 위원회(IOC)의 역할에 대해 설명하라."
query = "스포츠 대회에서 프로 선수의 참가에 대한 논란과 그 변화를 설명하라."
query = "월드컵과 같은 국제 축구 대회의 개최 과정과 인기 요인을 설명하라"
results = vector_store.similarity_search_with_score(
    query=query,  #찾을 질문
    k=5,   # 몇개 문서를 찾을지 지정
)

In [21]:
for result in results:
    print(result[1], result[0].page_content[:200])

0.39949131916007513 국제 올림픽 위원회
올림픽 활동이란 많은 수의 국가, 국제 경기 연맹과 협회 • 미디어 파트너를 맺기 • 선수, 직원, 심판, 모든 사람과 기관이 올림픽 헌장을 지키는 것을 말한다. 국제올림픽위원회(IOC)는 모든 올림픽 활동을 통솔하는 단체로서, 올림픽 개최 도시 선정, 계획 감독, 종목 변경, 스폰서 및 방송권 계약 체결 등의 권리가 있다. 올림픽 활동
0.3842431830894739 올림픽(영어: Olympic Games, 프랑스어: Jeux olympiques)은 전 세계 각 대륙 각국에서 모인 수천 명의 선수가 참가해 여름과 겨울에 스포츠 경기를 하는 국제적인 대회이다. 전 세계에서 가장 큰 지구촌 최대의 스포츠 축제인 올림픽은 세계에서 가장 인지도있는 국제 행사이다. 올림픽은 2년마다 하계 올림픽과 동계 올림픽이 번갈아 열리며, 
0.3666456940719483 올림픽은 국제경기연맹(IF), 국가 올림픽 위원회(NOC), 각 올림픽의 위원회(예-벤쿠버동계올림픽조직위원회)로 구성된다. 의사 결정 기구인 IOC는 올림픽 개최 도시를 선정하며, 각 올림픽 대회마다 열리는 올림픽 종목도 IOC에서 결정한다. 올림픽 경기 개최 도시는 경기 축하 의식이 올림픽 헌장에 부합하도록 조직하고 기금을 마련해야 한다. 올림픽 축하 행
0.34671767721031144 1896년에 그리스 아테네에서 제 1회 올림픽이 열렸다. 이때부터 IOC는 올림픽 운동의 감독 기구가 되었으며, 조직과 활동은 올림픽 헌장을 따른다. 오늘날 전 세계 대부분의 국가에서 올림픽 메달은 매우 큰 영예이며, 특히 올림픽 금메달리스트는 국가 영웅급의 대우를 받으며 스포츠 스타가 된다. 국가별로 올림픽 메달리스트들에게 지급하는 포상금도 크다. 대부분
0.33973791503721296 올림픽은 거의 모든 국가가 참여할 정도로 규모가 커졌다. 하계 올림픽은 33개의 종목과 약 400개의 세부종목에서 13,000명이 넘는 선수들이 겨루고 그중 각 종목별 1, 2

## MMR(최대 한계 관련성-Maximal Marginal Relevance) 알고리즘 적용
- 관련성과 다양성의 균형 조절
- 수학적 정의
$$
   \text{MMR} = \lambda \cdot \text{Sim}(d, Q) - (1 - \lambda) \cdot \max_{d' \in D'} \text{Sim}(d, d')
$$

 - $\text{Sim}(d, Q)$: 문서 $d$와 쿼리 $\text{Q}$ 사이의 유사성. (문서 유사성 계산)  (질문 - 문서 간의 유사성)
- $\max_{d' \in D'} \text{Sim}(d, d')$: 문서 $d$와 이미 선택된 문서 집합 $D'$ 중 가장 유사한 문서와의 유사성. (문서 다양성 계산)  (문서-문서 간의 유사성 = 다양성)
- $\lambda$: 유사성과 다양성의 중요도를 조절하는 매개변수(parameter) (0-1 사이의 수) -> 1에 가까우면 유사성을 더 보겠음/ 0에 가까우면 다양성을 더 보겠음

In [23]:
query = "동계 올림픽에 대해 설명해줘"
mmr_result = vector_store.max_marginal_relevance_search(
    query=query,
    k=5,    # 최종 결과 문서 개수
    fetch_k=20,   #처음 검색할 문서 개수
    lambda_mult=0.9   # 1에 가까울수록 유사성 0에 가까울수록 다양성을 최대화.
)
for result in mmr_result:
    print(result.page_content[:200])
    print("--------------------")

하계올림픽
--------------------
동계올림픽
동계 올림픽은 눈과 얼음을 이용하는 스포츠들을 모아 이루어졌으며 하계 올림픽 때 실행하기 불가능한 종목들로 구성되어 있다. 피겨스케이팅, 아이스하키는 각각 1908년과 1920년에 하계올림픽 종목으로 들어가 있었다. IOC는 다른 동계 스포츠로 구성된 새로운 대회를 만들고 싶어 했고, 로잔에서 열린 1921년 올림픽 의회에서 겨울판 올림픽을 열기
--------------------
올림픽
--------------------
오늘날의 올림픽
1896년 대회때는 14개국에서 241명의 선수단이 참가했지만 2008년 하계 올림픽때는 204개국에서 10,500명의 선수가 참가하는 등 세계적인 대회로 변모했다. 동계 올림픽의 규모는 하계 올림픽 규모보다 작다. 예를 들면 2006 토리노 동계 대회때는 80개국에서 2,508명의 선수가 참가했으며 82개 세부종목이 있었고, 2008 베이
--------------------
고대올림픽
--------------------
